# Compare Normal GraphRAG vs Optimized GraphRAG

This notebook measures whether the optimized capsule approach is cheaper/faster than normal GraphRAG on the same dataset slice.

Run order:

1. Run setup.
2. Run exactly one dataset block.
3. Run indexing.
4. Run parallel evaluation.
5. Run visualizations.

Ollama prerequisites:

```bash
ollama pull nomic-embed-text
ollama pull qwen3:4b-instruct
```

Ollama decides actual GPU placement. The notebook passes GPU-friendly model options and keeps models warm where supported.

In [ ]:
from __future__ import annotations

import json
import math
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.adapters.datasets import get_dataset_adapter
from src.adapters.embedding import (
    CachingEmbeddingAdapter, LlamaCppEmbeddingAdapter, LlamaCppEmbeddingConfig,
    OllamaEmbeddingAdapter, OllamaEmbeddingConfig, local_embedding_cache_namespace,
)
from src.adapters.llm import OllamaLLM, OllamaLLMConfig
from src.adapters.storage.graph_store import InMemoryGraphStore
from src.adapters.storage.vector_store import InMemoryVectorStore, NumpyVectorStore
from src.compute_eval import InMemoryComputeTracker
from src.evaluation import (
    CachedEvaluationRunner, EvaluationCache, answer_kind, context_answer_recall,
    fingerprint_bundle, fingerprint_run, normalize_text, summarize_metrics,
)
from src.g_rag import GraphRAG, GraphRAGConfig
from src.optimized_grag import OptimizedGraphRAG, OptimizedGraphRAGConfig
from src.optimized_grag.models import TrainedStructuralEncoder
from scripts.run_grag_evaluation import (
    DATASETS, _retrieval_configs, _source_fingerprint,
    _structural_artifact_fingerprint,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch device: {DEVICE}")

## Runtime Settings

`RUN_GENERATION = False` compares retrieval and graph-context efficiency without LLM answer noise.

Set it to `True` when you also want generated-answer timing and answer-hit checks. In that mode both systems use `qwen3:4b-instruct`.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
LLAMA_CPP_BASE_URL = "http://127.0.0.1:11435"
EMBEDDING_BACKEND = "llama_cpp"  # start scripts/start_batched_embedding_server.ps1 first
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen3:4b-instruct"

RUN_GENERATION = False
MAX_RETRIEVAL_HOPS = 3  # one shared query-time ceiling for both systems
MAX_WORKERS = 1  # safe default for 6 GB GPUs; raise only after measuring VRAM
OLLAMA_KEEP_ALIVE = "30m"
OLLAMA_OPTIONS = {"num_gpu": 999}
EMBED_REQUEST_SIZE = 128  # server request size; corpus checkpoints remain 2,048

if EMBEDDING_BACKEND == "llama_cpp":
    base_embedding_adapter = LlamaCppEmbeddingAdapter(LlamaCppEmbeddingConfig(
        model=EMBEDDING_MODEL, base_url=LLAMA_CPP_BASE_URL,
        timeout_seconds=180.0, batch_size=EMBED_REQUEST_SIZE, max_concurrent_requests=8,
    ))
    EMBEDDING_NAMESPACE = local_embedding_cache_namespace(EMBEDDING_MODEL)
else:
    base_embedding_adapter = OllamaEmbeddingAdapter(OllamaEmbeddingConfig(
        model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL, keep_alive=OLLAMA_KEEP_ALIVE,
        options=OLLAMA_OPTIONS, timeout_seconds=180.0, batch_size=16,
    ))
    EMBEDDING_NAMESPACE = local_embedding_cache_namespace(EMBEDDING_MODEL)
embedding_adapter = CachingEmbeddingAdapter(
    base_embedding_adapter,
    cache_path=ROOT / "artifacts" / "cache" / "embeddings.sqlite3",
    namespace=EMBEDDING_NAMESPACE,
    model_name=EMBEDDING_MODEL,
)
llm = OllamaLLM(
    OllamaLLMConfig(
        model=LLM_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
        num_predict=160,
        keep_alive=OLLAMA_KEEP_ALIVE,
        default_options=OLLAMA_OPTIONS,
        timeout_seconds=180.0,
    )
)

# Warm the embedding model once so indexing timings are less dominated by model load.
embedding_adapter.embed_texts(["warmup"])
if RUN_GENERATION:
    llm.generate("Reply with one word: ready")

{"embedding_model": EMBEDDING_MODEL, "llm_model": LLM_MODEL, "max_workers": MAX_WORKERS, "embedding_request_size": EMBED_REQUEST_SIZE, "max_retrieval_hops": MAX_RETRIEVAL_HOPS}

## Choose One Dataset

Run exactly one dataset block. Each block has conservative defaults so the notebook is practical on a laptop.

In [ ]:
METAQA_EVAL_HOP = 1  # chooses only the held-out report slice, never the runtime model
ACTIVE_DATASET = dict(DATASETS[f"metaqa_{METAQA_EVAL_HOP}hop"])
runtime_configs = _retrieval_configs(ACTIVE_DATASET, MAX_RETRIEVAL_HOPS)
ACTIVE_DATASET["normal_config"] = asdict(runtime_configs["normal_grag"])
ACTIVE_DATASET["optimized_config"] = asdict(runtime_configs["optimized_grag"])
ACTIVE_DATASET["index_native_graph_for_optimized"] = True
ACTIVE_DATASET

In [ ]:
ACTIVE_DATASET = {
    "label": "2WikiMultiHopQA train sample",
    "artifact_name": "2wiki_train_sample",
    "adapter_name": "2wiki",
    "adapter_kwargs": {"split": "train"},
    "limit_documents": 800,
    "limit_eval_samples": 200,
    "max_eval_questions": 40,
    "index_native_graph_for_optimized": False,
    "normal_config": {
        "chunk_size": 850,
        "chunk_overlap": 100,
        "top_k": 5,
        "graph_depth": MAX_RETRIEVAL_HOPS,
        "graph_limit": 80,
    },
    "optimized_config": {
        "chunk_size": 850,
        "chunk_overlap": 100,
        "max_region_nodes": 700,
        "max_capsule_nodes": 180,
        "max_capsule_edges": 650,
        "anchors_per_region": 4,
        "max_capsules_per_region": 4,
        "first_stage_candidate_capsules": 100,
        "top_k_capsules": 10,
        "max_candidate_paths": 200,
        "max_path_length": MAX_RETRIEVAL_HOPS,
        "seed_expansion_depth": MAX_RETRIEVAL_HOPS,
        "final_paths_to_llm": 12,
    },
}
ACTIVE_DATASET

In [ ]:
ACTIVE_DATASET = {
    "label": "MultiHop-RAG train sample",
    "artifact_name": "multihop_rag_train_sample",
    "adapter_name": "multihop_rag",
    "adapter_kwargs": {"split": "train"},
    "limit_documents": 1_000,
    "limit_eval_samples": 200,
    "max_eval_questions": 40,
    "index_native_graph_for_optimized": False,
    "normal_config": {
        "chunk_size": 850,
        "chunk_overlap": 100,
        "top_k": 5,
        "graph_depth": MAX_RETRIEVAL_HOPS,
        "graph_limit": 80,
    },
    "optimized_config": {
        "chunk_size": 850,
        "chunk_overlap": 100,
        "max_region_nodes": 700,
        "max_capsule_nodes": 180,
        "max_capsule_edges": 650,
        "anchors_per_region": 4,
        "max_capsules_per_region": 4,
        "first_stage_candidate_capsules": 100,
        "top_k_capsules": 10,
        "max_candidate_paths": 200,
        "max_path_length": MAX_RETRIEVAL_HOPS,
        "seed_expansion_depth": MAX_RETRIEVAL_HOPS,
        "final_paths_to_llm": 12,
    },
}
ACTIVE_DATASET

## Load Dataset And Build Systems

In [ ]:
if "ACTIVE_DATASET" not in globals():
    raise RuntimeError("Run exactly one dataset block first.")

bundle = get_dataset_adapter(
    ACTIVE_DATASET["adapter_name"],
    **ACTIVE_DATASET.get("adapter_kwargs", {}),
).load(
    limit_documents=ACTIVE_DATASET.get("limit_documents"),
    limit_eval_samples=ACTIVE_DATASET.get("limit_eval_samples"),
)
eval_samples = bundle.eval_samples[: ACTIVE_DATASET["max_eval_questions"]]

normal_tracker = InMemoryComputeTracker(log_stages=False, namespace="normal")
optimized_tracker = InMemoryComputeTracker(log_stages=False, namespace="optimized")

projector_dir = ROOT / "artifacts" / "rgcn" / ACTIVE_DATASET["artifact_name"]
structural_dir = ROOT / "artifacts" / "rgcn" / ACTIVE_DATASET.get("structural_artifact_name", ACTIVE_DATASET["artifact_name"])
structural_encoder = None
artifact_metadata_path = structural_dir / "capsule_rgcn_metadata.json"
artifact_manifest_path = structural_dir / "capsule_rgcn_embeddings.npz"
if artifact_metadata_path.exists():
    artifact_metadata = json.loads(artifact_metadata_path.read_text(encoding="utf-8"))
    projection_manifest = str(artifact_metadata.get("projection_manifest") or "")
    if projection_manifest:
        artifact_manifest_path = structural_dir / projection_manifest.split(":", 1)[0]
if artifact_manifest_path.exists():
    structural_encoder = TrainedStructuralEncoder(structural_dir, projector_artifact_dir=projector_dir)
    print(f"using structural artifacts from {structural_dir} and projector from {projector_dir}")

normal_config = GraphRAGConfig(**ACTIVE_DATASET.get("normal_config", {}))
corpus_hash = fingerprint_bundle(bundle, [])
normal_vector_index_key = fingerprint_run({
    "corpus": corpus_hash, "embedding_namespace": EMBEDDING_NAMESPACE,
    "chunk_size": normal_config.chunk_size,
    "chunk_overlap": normal_config.chunk_overlap,
    "store": "numpy_exact_cosine_v1",
})
normal_vector_index_dir = ROOT / "artifacts" / "cache" / "vector_indexes" / normal_vector_index_key
normal_grag = GraphRAG(
    graph_store=InMemoryGraphStore(),
    vector_store=NumpyVectorStore(cache_directory=normal_vector_index_dir),
    embedding_adapter=embedding_adapter,
    llm=llm,
    instrumentation=normal_tracker,
    config=normal_config,
)
optimized_grag = OptimizedGraphRAG(
    graph_store=InMemoryGraphStore(),
    vector_store=InMemoryVectorStore(),
    embedding_adapter=embedding_adapter,
    llm=llm,
    instrumentation=optimized_tracker,
    config=OptimizedGraphRAGConfig(**ACTIVE_DATASET.get("optimized_config", {})),
    structural_encoder=structural_encoder,
)

{
    "dataset": bundle.name,
    "documents": len(bundle.documents),
    "eval_samples": len(eval_samples),
    "native_graph_nodes": len(bundle.graph_nodes),
    "native_graph_edges": len(bundle.graph_edges),
}

## Index Both Approaches

Normal GraphRAG indexes all source chunks into a persistent dense index and uses the authoritative dataset graph when available.

Optimized GraphRAG indexes the same documents, then builds capsules and a capsule router. For graph-native datasets like MetaQA, it can also seed native dataset graph nodes/edges.

In [ ]:
index_rows = []

started = time.perf_counter()
if ACTIVE_DATASET.get("index_native_graph_for_optimized", True) and bundle.graph_nodes and bundle.graph_edges:
    normal_report = normal_grag.index_dataset_bundle(
        bundle,
        include_documents=bool(ACTIVE_DATASET.get("normal_index_documents", True)),
    )
else:
    normal_report = normal_grag.index_documents(bundle.documents)
normal_index_s = time.perf_counter() - started
if (
    bundle.documents
    and bool(ACTIVE_DATASET.get("normal_index_documents", True))
    and normal_report.vectors == 0
):
    raise RuntimeError(
        "Invalid production comparison: normal GraphRAG indexed zero source vectors. "
        "Restart the kernel and rerun all cells so the canonical dataset config and "
        "persistent NumpyVectorStore are active."
    )
index_rows.append(
    {
        "system": "normal_grag",
        "index_seconds": normal_index_s,
        "documents": normal_report.documents,
        "chunks": normal_report.chunks,
        "graph_nodes": normal_report.graph_nodes,
        "graph_edges": normal_report.graph_edges,
        "vectors": normal_report.vectors,
        "capsules": 0,
        "capsule_edges": 0,
        "trained_capsule_match_rate": None,
    }
)

started = time.perf_counter()
if ACTIVE_DATASET.get("index_native_graph_for_optimized", True):
    optimized_report = optimized_grag.index_dataset_bundle(
        bundle,
        include_documents=not ACTIVE_DATASET.get("native_graph_only", False),
    )
else:
    optimized_report = optimized_grag.index_documents(bundle.documents)
optimized_index_s = time.perf_counter() - started
trained_capsule_match_rate = None
if structural_encoder is not None and getattr(structural_encoder, "capsule_vectors", None):
    current_capsule_ids = [capsule.id for capsule in optimized_grag.capsule_index.capsules()]
    if current_capsule_ids:
        trained_capsule_match_rate = sum(
            capsule_id in structural_encoder.capsule_vectors for capsule_id in current_capsule_ids
        ) / len(current_capsule_ids)
index_rows.append(
    {
        "system": "optimized_grag",
        "index_seconds": optimized_index_s,
        "documents": optimized_report.documents,
        "chunks": optimized_report.chunks,
        "graph_nodes": optimized_report.graph_nodes,
        "graph_edges": optimized_report.graph_edges,
        "vectors": optimized_report.vectors,
        "capsules": optimized_report.capsules,
        "capsule_edges": optimized_report.capsule_edges,
        "trained_capsule_match_rate": trained_capsule_match_rate,
    }
)

index_df = pd.DataFrame(index_rows)
index_df

## Corpus Answer Coverage

This checks whether the indexed document slice contains the gold answer strings at all. If coverage is low, both systems are being evaluated on impossible or weakly grounded questions.


In [ ]:
corpus_text = normalize_text("\n".join(document.text for document in bundle.documents))
coverage_rows = []
for sample in eval_samples:
    coverage_rows.append(
        {
            "sample_id": sample.id,
            "sample_type": answer_kind(sample.answers),
            "answer_in_indexed_documents": float(any(normalize_text(answer) in corpus_text for answer in sample.answers if answer)),
        }
    )
coverage_df = pd.DataFrame(coverage_rows)
coverage_summary = coverage_df.groupby("sample_type")["answer_in_indexed_documents"].mean().to_frame("coverage")
display(coverage_summary)
answer_coverage = coverage_df["answer_in_indexed_documents"].dropna().mean()
minimum_coverage = ACTIVE_DATASET.get("min_answer_coverage")
if minimum_coverage is not None and answer_coverage < minimum_coverage:
    raise RuntimeError(
        f"Invalid retrieval benchmark: answer coverage {answer_coverage:.1%} "
        f"is below required {minimum_coverage:.1%}. Index the complete corpus first."
    )
answer_coverage


## Parallel Query Evaluation

The main retrieval-quality proxy is `context_answer_hit`: whether any gold answer string appears in the retrieved context. This avoids making the comparison depend on LLM answer style.

In [ ]:
# Pre-cache every evaluation query once, outside per-system latency. Without
# this, the first system pays Ollama inference and the second gets cache hits.
for start in range(0, len(eval_samples), 256):
    embedding_adapter.embed_texts([sample.question for sample in eval_samples[start : start + 256]])

evaluation_runner = CachedEvaluationRunner(
    cache=EvaluationCache(root=ROOT / "artifacts" / "cache" / "evaluations"),
    max_workers=MAX_WORKERS,
)
bundle_hash = fingerprint_bundle(bundle, eval_samples)
source_hash = _source_fingerprint()
structural_hash = _structural_artifact_fingerprint(
    ACTIVE_DATASET.get("structural_artifact_name", ACTIVE_DATASET["artifact_name"]),
    ACTIVE_DATASET["artifact_name"],
)
evaluation_name = ACTIVE_DATASET.get("report_name", ACTIVE_DATASET["artifact_name"])
result_rows = []
for system, pipeline, system_config in [
    ("normal_grag", normal_grag, ACTIVE_DATASET.get("normal_config", {})),
    ("optimized_grag", optimized_grag, ACTIVE_DATASET.get("optimized_config", {})),
]:
    run_hash = fingerprint_run({
        "schema": 4, "bundle": bundle_hash, "source": source_hash, "system": system,
        "structural_artifacts": structural_hash if system == "optimized_grag" else None,
        "config": system_config, "embedding_model": EMBEDDING_MODEL,
        "llm_model": LLM_MODEL if RUN_GENERATION else None,
        "run_generation": RUN_GENERATION,
    })
    execute = (
        (lambda sample, p=pipeline: p.answer(sample.question))
        if RUN_GENERATION else
        (lambda sample, p=pipeline: p.retrieve(sample.question))
    )
    records, metric_rows = evaluation_runner.run(
        namespace=f"{evaluation_name}/{system}",
        fingerprint=run_hash, system=system, samples=eval_samples, execute=execute,
    )
    for record, row in zip(records, metric_rows):
        details = record.details
        row.update({
            "sample_type": row["answer_kind"],
            "context_answer_hit": bool(row["context_answer_recall"] or False),
            "generated_answer_hit": row["exact_match"],
            "selected_units": row["selected_capsules"] or details.get("vector_results", 0),
            "graph_nodes": details.get("graph_nodes", 0),
            "graph_edges": details.get("graph_edges", 0),
            "evidence_paths": details.get("evidence_paths", 0),
            "seed_nodes": row["linked_seeds"],
        })
        result_rows.append(row)

results_df = pd.DataFrame(result_rows)
results_df.head()

## Summary Metrics

In [ ]:
def p95(values) -> float:
    values = sorted(float(value) for value in values)
    if not values:
        return 0.0
    index = min(len(values) - 1, math.ceil(0.95 * len(values)) - 1)
    return values[index]


summary_rows = []
for system, group in results_df.groupby("system"):
    summary_rows.append(
        {
            "system": system,
            "queries": len(group),
            "mean_latency_s": group["latency_seconds"].mean(),
            "p50_latency_s": group["latency_seconds"].median(),
            "p95_latency_s": p95(group["latency_seconds"]),
            "context_answer_hit_rate": group["context_answer_hit"].mean(),
            "span_context_answer_hit_rate": group[group["sample_type"] == "span"]["context_answer_hit"].mean(),
            "mean_context_chars": group["context_chars"].mean(),
            "mean_selected_units": group["selected_units"].mean(),
            "mean_graph_nodes": group["graph_nodes"].mean(),
            "mean_graph_edges": group["graph_edges"].mean(),
            "mean_evidence_paths": group["evidence_paths"].mean(),
        }
    )
summary_df = pd.DataFrame(summary_rows).set_index("system")

if {"normal_grag", "optimized_grag"} <= set(summary_df.index):
    normal = summary_df.loc["normal_grag"]
    optimized = summary_df.loc["optimized_grag"]
    relative = {
        "latency_speedup_normal_over_optimized": normal["mean_latency_s"] / max(optimized["mean_latency_s"], 1e-12),
        "context_char_reduction": 1.0 - optimized["mean_context_chars"] / max(normal["mean_context_chars"], 1e-12),
        "graph_node_reduction": 1.0 - optimized["mean_graph_nodes"] / max(normal["mean_graph_nodes"], 1e-12),
        "context_hit_delta": optimized["context_answer_hit_rate"] - normal["context_answer_hit_rate"],
    }
else:
    relative = {}

display(index_df)
display(summary_df)
relative

## Visualizations

In [ ]:
systems = ["normal_grag", "optimized_grag"]
colors = ["#4C78A8", "#F58518"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f"GraphRAG vs Optimized GraphRAG - {ACTIVE_DATASET['label']}", fontsize=16)

def bar(ax, values, title, ylabel):
    ax.bar(systems, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=15)
    for idx, value in enumerate(values):
        ax.text(idx, value, f"{value:.3g}", ha="center", va="bottom")

index_by_system = index_df.set_index("system")
bar(axes[0, 0], [index_by_system.loc[s, "index_seconds"] for s in systems], "Index Time", "seconds")
bar(axes[0, 1], [summary_df.loc[s, "mean_latency_s"] for s in systems], "Mean Query Latency", "seconds")
bar(axes[0, 2], [summary_df.loc[s, "p95_latency_s"] for s in systems], "P95 Query Latency", "seconds")
bar(axes[1, 0], [summary_df.loc[s, "context_answer_hit_rate"] for s in systems], "Context Answer Hit Rate", "rate")
bar(axes[1, 1], [summary_df.loc[s, "mean_context_chars"] for s in systems], "Mean Context Size", "characters")
bar(axes[1, 2], [summary_df.loc[s, "mean_graph_nodes"] for s in systems], "Mean Graph Nodes In Context", "nodes")

plt.tight_layout()
plt.show()

## Stage Timing Breakdown

In [ ]:
def stage_totals_df(snapshot: dict, system: str) -> pd.DataFrame:
    rows = [
        {"system": system, "stage": stage, "total_ms": total_ms}
        for stage, total_ms in snapshot.get("stage_totals_ms", {}).items()
    ]
    return pd.DataFrame(rows)


stages_df = pd.concat(
    [
        stage_totals_df(normal_tracker.snapshot(), "normal_grag"),
        stage_totals_df(optimized_tracker.snapshot(), "optimized_grag"),
    ],
    ignore_index=True,
)
display(stages_df.sort_values(["system", "total_ms"], ascending=[True, False]).head(30))

if not stages_df.empty:
    top = stages_df.sort_values("total_ms", ascending=False).head(12)
    plt.figure(figsize=(12, 5))
    labels = top["system"] + "\n" + top["stage"]
    plt.bar(labels, top["total_ms"], color="#72B7B2")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("total ms")
    plt.title("Top Timed Stages")
    plt.tight_layout()
    plt.show()

## Save Report

In [ ]:
report_dir = ROOT / "reports" / "grag_comparison" / ACTIVE_DATASET.get("report_name", ACTIVE_DATASET["artifact_name"])
report_dir.mkdir(parents=True, exist_ok=True)

index_df.to_csv(report_dir / "index_metrics.csv", index=False)
results_df.to_csv(report_dir / "query_results.csv", index=False)
summary_df.to_csv(report_dir / "summary_metrics.csv")
if "coverage_df" in globals():
    coverage_df.to_csv(report_dir / "coverage_metrics.csv", index=False)
if "stages_df" in globals() and not stages_df.empty:
    stages_df.to_csv(report_dir / "stage_totals.csv", index=False)

with (report_dir / "run_config.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "dataset": ACTIVE_DATASET,
            "embedding_model": EMBEDDING_MODEL,
            "llm_model": LLM_MODEL,
            "run_generation": RUN_GENERATION,
            "max_workers": MAX_WORKERS,
            "max_retrieval_hops": MAX_RETRIEVAL_HOPS,
            "relative": relative,
        },
        f,
        indent=2,
    )

report_dir